# Step 3b: LoRA, given a fair chance
### Neeraj, MMAI 894 (router fine-tuning)

**Why this notebook:** in notebook 03, LoRA lost to ordinary fine-tuning, but its learning curve at the larger learning step (3e-4) was still climbing steeply at epoch 30. So it may have run out of time rather than ability. Before reporting LoRA as a negative result, we give it more room, changing as little else as possible.

| Name | What it is |
|---|---|
| `baseline` | notebook 02's model: ordinary fine-tuning, type and escalation heads (already run, read from the repo) |
| `lora_3e4_30ep` | notebook 03's LoRA at 3e-4, 30 epochs (already run, read from the repo; it also had a language head) |
| `lora_3e4_60ep` | LoRA at 3e-4 for **60 epochs**, type and escalation heads only (like the baseline) |
| `lora_1e3_60ep` | LoRA at a **bigger step, 1e-3**, 60 epochs, type and escalation heads only |

Same folds, same first-message input, same three seeds, same LoRA size (rank 8 on the attention query and value layers). Scores are saved after every epoch, so the 60-epoch runs also show what happens at epoch 30.

**What you need to do:** GPU on (Accelerator: GPU T4), Internet on, run all cells. The long cell takes about 30 to 35 minutes; if the session stops, run it again and it carries on. At the end, download the files listed in the last cell into `results/step03b`.

Everything shown is computed from saved files; this text holds no results.

## Step 0: Get the latest code and the LoRA library

In [ ]:
import importlib
import os
import shutil
import sys

REPO_URL = "https://github.com/neerajmehla28/northstar-finetuning"
REPO_DIR = "/kaggle/working/northstar-finetuning"

%cd /kaggle/working
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone -q {REPO_URL} {REPO_DIR}          # download the repo (public, so no login is needed)
!pip install -q peft                         # LoRA library
!pip uninstall -y -q torchao                 # Kaggle's older torchao makes the new peft refuse to run; we do not use it

%cd {REPO_DIR}
sys.path.insert(0, "src")
importlib.invalidate_caches()
for name in [m for m in sys.modules if m.startswith("northstar")]:
    del sys.modules[name]

assert os.path.exists("results/cv_old_recipe/first_message"), "notebook 02's results are missing from the repo"
assert os.path.exists("results/cv_step03/lora_3e4/first_message"), "notebook 03's results are missing from the repo: commit and push results/cv_step03 first"
!git log --oneline -3

## Step 1: Check the machine and the data

In [ ]:
import torch
import pandas as pd
from northstar.finetune import protocol as P
from northstar.finetune.train_cv import TrainConfig, run_cv

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
assert torch.cuda.is_available(), "No GPU found. Set Accelerator to GPU T4 in the notebook settings."

df = P.load_all()                 # refuses to run if the dataset file is not the exact team file
pool = P.pool(df)                 # the 160 chats; the 40 test chats are not in here
print("Pool:", len(pool), "chats")

## Step 2: Settings
Only the number of epochs and the learning step change; the heads are now the same as the baseline's, so this is LoRA against ordinary fine-tuning and nothing else.

In [ ]:
OUT = "/kaggle/working/cv_step03b"
RESULTS = "/kaggle/working/results_03b"
os.makedirs(RESULTS, exist_ok=True)

VARIANT = "first_message"
SEEDS = P.CV_SEEDS
HEADS = ("type", "escalation")             # the same two heads as the baseline

CONFIGS = {
    "lora_3e4_60ep": TrainConfig(epochs=60, heads=HEADS, lora=True, lr=3e-4),
    "lora_1e3_60ep": TrainConfig(epochs=60, heads=HEADS, lora=True, lr=1e-3),
}
for name, c in CONFIGS.items():
    print(f"{name:14s} lora={c.lora} lr={c.lr} epochs={c.epochs} heads={c.heads} rank={c.lora_r}")

## Step 3: Smoke test and time estimate
One fold, one seed, two epochs per model: proves the pipeline runs before the long cell. The estimate includes loading the model each time, so the real run is usually a little shorter.

In [ ]:
import time

total_s = 0.0
for name, cfg in CONFIGS.items():
    t0 = time.time()
    run_cv(cfg, VARIANT, f"/kaggle/working/smoke/{name}", folds=[0], seeds=[11], epochs=2, verbose=False)
    s = time.time() - t0
    total_s += s
    print(f"{name:14s} smoke test OK: {s:.0f}s for 2 epochs")

n_runs = len(CONFIGS) * len(SEEDS) * P.N_FOLDS
print(f"Full run = {n_runs} runs x 60 epochs; rough estimate {total_s / len(CONFIGS) / 2 * 60 * n_runs / 60:.0f} minutes (upper bound).")

## Step 4: The full cross-validation
This is the long cell; run it again if the session is interrupted (finished runs are kept).

In [ ]:
for name, cfg in CONFIGS.items():
    print(f"=== {name}")
    run_cv(cfg, VARIANT, f"{OUT}/{name}", seeds=SEEDS)
print("All runs finished.")

## Step 5: Learning curves
Every chat gets a score from models that never saw it. Each model is judged at its own best epoch by the same rule as before (smooth the average curve over three epochs, take the best).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from northstar.finetune import cv_analysis as A
from northstar.finetune import metrics as M

HEADS_SCORED = ("type", "escalation")      # the language head of notebook 03's LoRA run is ignored here
sources = {"baseline": "results/cv_old_recipe", "lora_3e4_30ep": "results/cv_step03/lora_3e4"}
sources.update({name: f"{OUT}/{name}" for name in CONFIGS})

runs, oof, curves, best_epoch = {}, {}, {}, {}
for name, folder in sources.items():
    runs[name] = A.load_variant(folder, VARIANT)
    oof[name] = A.assemble(runs[name], pool, HEADS_SCORED)
    curves[name] = A.epoch_curves(oof[name], pool)
    best_epoch[name], _ = A.pick_epoch(curves[name])
print("Best epoch per model:", best_epoch)

fig, ax = plt.subplots(figsize=(9, 4.8))
for name in sources:
    g = curves[name].groupby("epoch")["type_macro_f1"]
    ax.plot(g.mean().index, g.mean(), label=name)
    ax.fill_between(g.mean().index, g.mean() - g.std(), g.mean() + g.std(), alpha=0.12)
ax.axvline(30, color="grey", lw=0.8, ls=":")
ax.set_xlabel("epoch"); ax.set_ylabel("held-out macro-F1 (mean over seeds)"); ax.set_title("Request type: LoRA with more room"); ax.legend()
plt.tight_layout(); plt.savefig(f"{RESULTS}/learning_curves_03b.png", dpi=150); plt.show()

## Step 6: Results
Mean ± standard deviation over the three seeds at each model's best epoch, plus the new LoRA runs read at epoch 30 (to see what the extra 30 epochs bought). "Ensemble" averages the three seeds' scores before deciding.

In [ ]:
def row(n, epoch, label):
    s = A.summary_at_epoch(oof[n], pool, epoch)
    ps, et, ee = s["per_seed"], s["ensemble_type"], s["ensemble_escalation"]
    r = runs[n]
    trainable = int(np.mean([x["n_trainable"] for x in r])) if "n_trainable" in r[0] else None
    secs = float(np.mean([x["seconds"] for x in r])) if "seconds" in r[0] else None
    shown = {"model": label, "epoch": epoch,
             "type accuracy": f"{ps['type_accuracy'].mean():.3f} ± {ps['type_accuracy'].std():.3f}",
             "type macro-F1": f"{ps['type_macro_f1'].mean():.3f} ± {ps['type_macro_f1'].std():.3f}",
             "escalation recall": f"{ps['esc_recall'].mean():.3f} ± {ps['esc_recall'].std():.3f}",
             "escalation precision": f"{ps['esc_precision'].mean():.3f} ± {ps['esc_precision'].std():.3f}",
             "ensemble type acc": f"{et['accuracy']:.3f}",
             "trainable numbers": f"{trainable:,}" if trainable else "n/a", "seconds per run": f"{secs:.0f}" if secs else "n/a"}
    numeric = dict(model=label, epoch=epoch, trainable=trainable, seconds_per_run=secs,
                   **{f"{c}_mean": ps[c].mean() for c in ps.columns if c != "seed"},
                   **{f"{c}_sd": ps[c].std() for c in ps.columns if c != "seed"},
                   ensemble_type_accuracy=et["accuracy"], ensemble_type_macro_f1=et["macro_f1"],
                   ensemble_esc_recall=ee["recall"], ensemble_esc_precision=ee["precision"])
    return shown, numeric, s

summ, shown_rows, numeric_rows = {}, [], []
for n in sources:
    shown, numeric, summ[n] = row(n, best_epoch[n], n)
    shown_rows.append(shown); numeric_rows.append(numeric)
for n in CONFIGS:                                   # the same new runs, read at epoch 30
    shown, numeric, _ = row(n, 30, f"{n} @ epoch 30")
    shown_rows.append(shown); numeric_rows.append(numeric)
table = pd.DataFrame(shown_rows).set_index("model")
pd.DataFrame(numeric_rows).to_csv(f"{RESULTS}/cv_summary_step03b.csv", index=False)
table

## Step 7: Is any LoRA run really different from the baseline?
Intervals from resampling customers, and every LoRA run against the baseline on the same chats (exact McNemar). All comparisons are listed.

In [ ]:
y_type = pool["type_label"].values
customers = pool["customer_id"].values
correct = {n: summ[n]["ensemble_type_logits"].argmax(-1) == y_type for n in sources}

ci_rows = []
for n in sources:
    lo, hi = M.cluster_bootstrap(lambda idx, c=correct[n]: c[idx].mean(), customers, n_boot=2000)
    ci_rows.append(dict(model=n, ensemble_accuracy=correct[n].mean(), ci_low=lo, ci_high=hi))
ci = pd.DataFrame(ci_rows).set_index("model")

pair_rows = []
for n in [s for s in sources if s != "baseline"]:
    r = M.mcnemar_exact(correct[n], correct["baseline"])
    pair_rows.append(dict(model=n, versus="baseline", only_model_right=r["only_a_right"], only_baseline_right=r["only_b_right"], p_value=r["p_value"]))
pairs = pd.DataFrame(pair_rows)

ci.round(3).to_csv(f"{RESULTS}/accuracy_intervals_step03b.csv"); pairs.to_csv(f"{RESULTS}/comparisons_step03b.csv", index=False)
display(ci.round(3)); display(pairs.round(3))

## Step 8: Save the results
Download `cv_step03b.zip` (raw scores of every run), the two tables, the comparison file and the figure from `results_03b`, plus the notebook itself (File → Download Notebook), into `results/step03b`.

In [ ]:
shutil.make_archive(f"{RESULTS}/cv_step03b", "zip", OUT)
for name in sorted(os.listdir(RESULTS)):
    print(f"{name:40s} {os.path.getsize(os.path.join(RESULTS, name)) / 1024:8.0f} KB")

## What this notebook shows, and its limits
- **Shows:** whether LoRA catches up with ordinary fine-tuning when it gets twice the epochs or a bigger learning step, compared on the same folds, input, seeds and heads.
- **Limits:**
  - Trying more LoRA settings and keeping the best, on the same folds it is scored on, favours LoRA slightly; this is reported, not hidden.
  - Two settings only (learning step and epochs); LoRA rank, the layers it adapts and warmup are left as in notebook 03.
  - 160 chats and 3 seeds: small differences are inside the noise, which is why intervals and paired tests are shown.